# Experiment 6 — Dense Stock Graph + Model Grid

Runs every non-empty combination of {sector, correlation, co-mention} edges
across four graph models and three seeds, then adds text encoders.

Runtime -> Change runtime type -> **GPU** before running.

Results are written after every single run and completed cells are skipped
on restart, so if this session crashes, just re-run from the top.

In [ ]:
!git clone https://github.com/AdityaMelkote3004/capstone.git repo 2>/dev/null || echo "already cloned"
%cd repo
!git pull
!pip install -q torch_geometric
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PERSIST = '/content/drive/MyDrive/mmgtfff_experiment6'
os.makedirs(PERSIST, exist_ok=True)
# Symlink results into Drive so every completed run is durable.
!rm -rf results/experiment6_dense_graph
!ln -s {PERSIST} results/experiment6_dense_graph
print("results persist to", PERSIST)

In [ ]:
!pip install -q pytest
!python -m pytest tests/test_graph_edges.py tests/test_graph_models.py tests/test_graph_transformer_multi.py -q

In [ ]:
!python scripts/run_dense_graph_grid.py --pass 1

In [ ]:
!python scripts/run_dense_graph_grid.py --pass 2

In [ ]:
!python scripts/run_dense_graph_grid.py --pass 3

In [ ]:
!python scripts/make_experiment6_figures.py
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('results/experiment6_dense_graph/figures/*.png')):
    print(p); display(Image(p))

In [ ]:
import pandas as pd
df = pd.read_csv('results/experiment6_dense_graph/results.csv')
summary = (df.groupby(['model','edges'])['mcc']
             .agg(['mean','std','count'])
             .sort_values('mean', ascending=False))
print(summary.to_string())
print("
Baselines -- Exp1: +0.0922, Exp3 price-only: +0.0108, Exp4 sector GAT: +0.0278")
print("Constant predictors -- always-UP: acc 0.5129 / F1 0.6780, always-DOWN: acc 0.4871 / F1 0.0")